# ID Document Tamper Detection & Forensic Verification Pipeline
This notebook trains a forensic dual-stream CNN on MIDV-500, audits data integrity via SHA-256, and verifies semantic integrity via ICAO 9303 MRZ checksums.

In [ ]:
# Cell 1: Environment Setup & Hardware Check
!pip install -q timm albumentations opencv-python-headless

import os
import glob
import json
import random
import hashlib
import urllib.request
import zipfile
from collections import defaultdict
import numpy as np
import cv2
from PIL import Image, ImageChops, ImageEnhance
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device initialized: {device}")

In [ ]:
# Cell 2: MIDV-500 Ingestion & Resilient Fallback Engine
DATA_DIR = "./data/midv500"
PREP_DIR = "./data/prepared_dataset"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f"{PREP_DIR}/train/pristine", exist_ok=True)
os.makedirs(f"{PREP_DIR}/train/tampered", exist_ok=True)
os.makedirs(f"{PREP_DIR}/val/pristine", exist_ok=True)
os.makedirs(f"{PREP_DIR}/val/tampered", exist_ok=True)

# Sample subset from MIDV-500
SAMPLE_URL = "ftp://smartengines.com/midv-500/dataset/01_alb_id.zip"
ZIP_PATH = os.path.join(DATA_DIR, "01_alb_id.zip")

raw_files = glob.glob(f"{DATA_DIR}/**/*.tif", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)

if len(raw_files) < 10:
    print("Downloading MIDV-500 subset...")
    try:
        urllib.request.urlretrieve(SAMPLE_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Download and extraction complete.")
    except Exception as e:
        print(f"Direct FTP unreachable ({e}). Creating simulated MIDV-500 video clip directories for local execution...")
        for clip_idx in range(10):
            clip_path = os.path.join(DATA_DIR, f"clip_{clip_idx:02d}")
            os.makedirs(clip_path, exist_ok=True)
            for frame_idx in range(15):
                canvas = np.full((512, 512, 3), 230 - (clip_idx * 5), dtype=np.uint8)
                cv2.rectangle(canvas, (30, 30), (480, 480), (190, 190, 190), -1)
                cv2.putText(canvas, f"ID CARD #{1000+clip_idx}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (15, 15, 15), 2)
                cv2.putText(canvas, f"EXP: 2029-08-12", (50, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (20, 20, 20), 2)
                cv2.putText(canvas, f"FRAME {frame_idx:02d}", (50, 440), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (60, 60, 60), 1)
                cv2.imwrite(os.path.join(clip_path, f"frame_{frame_idx:02d}.jpg"), canvas)

all_images = glob.glob(f"{DATA_DIR}/**/*.tif", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)
print(f"Total raw frame images gathered: {len(all_images)}")

In [ ]:
# Cell 3: Grouped Data Partitioning & Synthetic Tamper Engine with SHA-256 Checks
def compute_sha256(filepath):
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

def apply_synthetic_tampering(img):
    tampered = img.copy()
    h, w, _ = tampered.shape
    tamper_mode = random.choice(['text_splice', 'copy_move', 'gaussian_blur'])
    
    if tamper_mode == 'text_splice':
        y, x = random.randint(80, h - 100), random.randint(50, w - 180)
        fake_digits = str(random.randint(10000000, 99999999))
        cv2.putText(tampered, fake_digits, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (20, 20, 20), 2)
    elif tamper_mode == 'copy_move':
        ph, pw = random.randint(30, 60), random.randint(60, 110)
        y1, x1 = random.randint(0, h - ph), random.randint(0, w - pw)
        patch = tampered[y1:y1+ph, x1:x1+pw].copy()
        y2, x2 = random.randint(0, h - ph), random.randint(0, w - pw)
        tampered[y2:y2+ph, x2:x2+pw] = patch
    else:
        y, x = random.randint(50, h - 100), random.randint(50, w - 150)
        tampered[y:y+50, x:x+80] = cv2.GaussianBlur(tampered[y:y+50, x:x+80], (19, 19), 0)
        
    return tampered

# Prevent frame leakage: partition strictly by directory / clip prefix
clip_map = defaultdict(list)
for path in all_images:
    clip_key = os.path.dirname(path)
    clip_map[clip_key].append(path)

unique_clips = list(clip_map.keys())
random.shuffle(unique_clips)
train_split_count = max(1, int(0.8 * len(unique_clips)))
train_clips = unique_clips[:train_split_count]
val_clips = unique_clips[train_split_count:]

def populate_split(clips, split_type):
    counter = 0
    for clip in clips:
        frames = sorted(clip_map[clip])
        # Sample distinct frames per clip to eliminate temporal redundancy
        sampled_frames = [frames[len(frames) // 2]] if len(frames) > 0 else []
        if len(frames) >= 4:
            sampled_frames.append(frames[len(frames) // 4])
            sampled_frames.append(frames[(3 * len(frames)) // 4])
            
        for f_path in sampled_frames:
            img = cv2.imread(f_path)
            if img is None:
                continue
            img = cv2.resize(img, (512, 512))
            
            # Write pristine sample
            pristine_dest = f"{PREP_DIR}/{split_type}/pristine/{counter}.jpg"
            cv2.imwrite(pristine_dest, img)
            
            # Write synthetic tampered sample
            tampered_img = apply_synthetic_tampering(img)
            tampered_dest = f"{PREP_DIR}/{split_type}/tampered/{counter}.jpg"
            cv2.imwrite(tampered_dest, tampered_img)
            
            # Cryptographic SHA-256 verification (guarantees bit difference)
            assert compute_sha256(pristine_dest) != compute_sha256(tampered_dest), f"Hash collision on index {counter}!"
            counter += 1

populate_split(train_clips, "train")
populate_split(val_clips, "val")
print(f"Partitioned dataset ready: {len(train_clips)} train clips, {len(val_clips)} val clips.")

In [ ]:
# Cell 4: Error Level Analysis (ELA) & Dataset Pipeline
def extract_ela(path, quality=90):
    im = Image.open(path).convert('RGB')
    temp_file = f"temp_{random.randint(1000, 9999)}.jpg"
    im.save(temp_file, 'JPEG', quality=quality)
    resaved = Image.open(temp_file)
    
    ela_im = ImageChops.difference(im, resaved)
    extrema = ela_im.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    if max_diff == 0:
        max_diff = 1
    scale_factor = 255.0 / max_diff
    ela_im = ImageEnhance.Brightness(ela_im).enhance(scale_factor * 0.5)
    
    if os.path.exists(temp_file):
        os.remove(temp_file)
    return ela_im

class ForensicDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.samples = []
        pristine_files = glob.glob(f"{root_dir}/{split}/pristine/*.jpg")
        tampered_files = glob.glob(f"{root_dir}/{split}/tampered/*.jpg")
        
        for p in pristine_files:
            self.samples.append((p, 0.0))
        for t in tampered_files:
            self.samples.append((t, 1.0))
            
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # RGB Stream
        rgb = cv2.imread(img_path)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (256, 256)).astype(np.float32) / 255.0
        
        # ELA Stream (Compression Residuals)
        ela = extract_ela(img_path)
        ela = ela.resize((256, 256))
        ela = np.array(ela).astype(np.float32) / 255.0
        
        # Concatenate 6 channels (3 RGB + 3 ELA)
        fused = np.concatenate([rgb, ela], axis=-1).transpose(2, 0, 1)
        return torch.tensor(fused, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

train_dataset = ForensicDataset(PREP_DIR, split="train")
val_dataset = ForensicDataset(PREP_DIR, split="val")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=False)
print(f"Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

In [ ]:
# Cell 5: Dual-Stream ResNet-18 Architecture & Training Loop
class DualStreamTamperResNet(nn.Module):
    def __init__(self):
        super(DualStreamTamperResNet, self).__init__()
        base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Expand input layer from 3 channels to 6 channels
        self.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = base_model.conv1.weight
            self.conv1.weight[:, 3:] = base_model.conv1.weight
            
        self.backbone = nn.Sequential(
            self.conv1,
            base_model.bn1,
            base_model.relu,
            base_model.maxpool,
            base_model.layer1,
            base_model.layer2,
            base_model.layer3,
            base_model.layer4,
            base_model.avgpool
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        logits = self.classifier(feat)
        return logits

model = DualStreamTamperResNet().to(device)
criterion = nn.BCEWithLogitsLoss() # Prevents numerical instability in standard BCELoss
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

EPOCHS = 8
print("Beginning training optimization...")
for epoch in range(EPOCHS):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        t_correct += (preds == labels).sum().item()
        t_total += imgs.size(0)
        
    scheduler.step()
    
    # Validation phase
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)
            
            v_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            v_correct += (preds == labels).sum().item()
            v_total += imgs.size(0)
            
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] | Train Loss: {t_loss/t_total:.4f} | Train Acc: {t_correct/t_total:.1%} | Val Loss: {v_loss/v_total:.4f} | Val Acc: {v_correct/v_total:.1%}")

torch.save(model.state_dict(), "tamper_model.pth")
print("Weights saved to tamper_model.pth")

In [ ]:
# Cell 6: ICAO 9303 Checksum Engine & End-to-End Hybrid Demonstration
def verify_mrz_checksum(field_str: str, check_digit: str) -> bool:
    """
    ICAO 9303 standard 7-3-1 Modulo-10 checksum algorithm.
    """
    weights = [7, 3, 1]
    total = 0
    for idx, ch in enumerate(field_str):
        if ch.isdigit():
            val = int(ch)
        elif ch.isalpha():
            val = ord(ch.upper()) - 55
        elif ch == '<':
            val = 0
        else:
            return False
        total += val * weights[idx % 3]
    return (total % 10) == int(check_digit)

def run_hybrid_inference(image_path, semantic_data, expected_checksum):
    model.eval()
    # 1. Visual / ELA stream
    rgb = cv2.imread(image_path)
    rgb_clean = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    rgb_tensor = cv2.resize(rgb_clean, (256, 256)).astype(np.float32) / 255.0
    
    ela = extract_ela(image_path)
    ela_resized = ela.resize((256, 256))
    ela_tensor = np.array(ela_resized).astype(np.float32) / 255.0
    
    fused = np.concatenate([rgb_tensor, ela_tensor], axis=-1).transpose(2, 0, 1)
    tensor_in = torch.tensor(fused, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logit = model(tensor_in).squeeze().item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        
    # 2. Checksum stream
    checksum_passed = verify_mrz_checksum(semantic_data, expected_checksum)
    
    # 3. Decision Matrix
    visual_flag = prob >= 0.5
    checksum_flag = not checksum_passed
    overall_forgery = visual_flag or checksum_flag
    
    fig, axs = plt.subplots(1, 2, figsize=(9, 4))
    axs[0].imshow(rgb_clean)
    axs[0].set_title(f"Input Frame (Tamper Prob: {prob*100:.1f}%)")
    axs[0].axis('off')
    
    axs[1].imshow(ela)
    axs[1].set_title("Extracted ELA Residuals")
    axs[1].axis('off')
    plt.show()
    
    print(f"================ VERIFICATION AUDIT ================")
    print(f"Visual Deep Learning Anomaly : {'DETECTED' if visual_flag else 'NONE'} ({prob:.4f})")
    print(f"ICAO 9303 MRZ Checksum Status: {'VALID' if checksum_passed else 'MISMATCH (TAMPER DETECTED)'}")
    print(f"Final Hybrid System Verdict  : {'REJECTED (FORGED DOCUMENT)' if overall_forgery else 'APPROVED (AUTHENTIC DOCUMENT)'}")
    print(f"====================================================")

# Run evaluation on a prepared validation sample
val_samples = glob.glob(f"{PREP_DIR}/val/tampered/*.jpg")
if val_samples:
    run_hybrid_inference(val_samples[0], semantic_data="980412", expected_checksum="1")